# 03 — Qualité des données et validation du calcul des apports

1. Contrôles qualité selon les 8 dimensions de la consigne (`ControleQualite`).
2. Calcul des apports (`CalculApports`) et **validation** : on compare nos apports à ceux publiés par l'Anses.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # pour importer le code de src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nutrihfr import config as C

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})
C.DOSSIER_FIGURES.mkdir(parents=True, exist_ok=True)
C.DOSSIER_TABLES.mkdir(parents=True, exist_ok=True)

In [ ]:
from nutrihfr.pipeline import ChargeurINCA3, ControleQualite, CalculApports
individus, consommations, apports_anses = ChargeurINCA3().charger()

## 1. Rapport qualité

In [ ]:
qualite = ControleQualite(individus, consommations).executer()
qualite.to_csv(C.DOSSIER_TABLES / "rapport_qualite.csv", index=False)
qualite

**Bilan** : les données sont de très bonne qualité. Deux points « à vérifier » seulement, sans conséquence sur les résultats :
- 5 adultes sans poids corporel : exclus seulement de l'analyse de sous-déclaration ;
- 10 lignes sans composition : elles ne comptent pas dans les apports (même choix que l'Anses).

Le contrôle « Atwater » vérifie la cohérence de la composition : l'énergie recalculée à partir des macronutriments (4 kcal/g de protéines et de glucides, 9 de lipides, 2 de fibres, 7 d'alcool) est proche de l'énergie indiquée pour 97 % des lignes. Les écarts restants s'expliquent par des composants non pris en compte (acides organiques, polyols).

## 2. Calcul des apports

Pour chaque individu (méthode de l'Anses) :

$$\text{apport journalier} = \frac{1}{\text{nb de jours}} \sum_{\text{lignes}} \frac{\text{qte\_conso\_pond}}{100} \times \text{composition pour 100 g}$$

`qte_conso_pond` = quantité × `R24_pond` : chaque jour compte selon qu'il est en semaine ou en week-end, pour représenter une semaine type.

In [ ]:
calcul = CalculApports(consommations, individus)
apports = calcul.par_individu()
apports.to_csv(C.DOSSIER_TRAITE / "apports_individuels.csv", index=False)
apports[["NOIND", "sexe", "classe_age"] + list(C.NUTRIMENTS)].head()

Exemple détaillé pour un individu, pour bien comprendre le calcul :

In [ ]:
exemple = consommations[consommations["NOIND"] == consommations["NOIND"].iloc[0]]
detail = exemple[["R24_num", "aliment_libelle_INCA3", "qte_conso", "R24_pond", "qte_conso_pond", "calcium"]].copy()
detail["calcium_apporte_mg"] = detail["qte_conso_pond"] / 100 * detail["calcium"]
display(detail.head(8))
n_jours = exemple["R24_nombre"].iloc[0]
print(f"Total sur {n_jours} jours : {detail['calcium_apporte_mg'].sum():.0f} mg "
      f"-> apport journalier = {detail['calcium_apporte_mg'].sum() / n_jours:.0f} mg/j")

## 3. Validation : nos apports vs les apports publiés par l'Anses

In [ ]:
comparaison = apports.merge(apports_anses, on="NOIND", suffixes=("", "_anses"))
validation = []
for n in C.NUTRIMENTS:
    ecart = (comparaison[n] - comparaison[n + "_anses"]).abs()
    validation.append({"nutriment": C.NUTRIMENTS[n][0], "corrélation": comparaison[n].corr(comparaison[n + "_anses"]),
                       "écart maximal": ecart.max(), "% individus identiques (écart < 0,01)": 100 * (ecart < 0.01).mean()})
validation = pd.DataFrame(validation)
validation.to_csv(C.DOSSIER_TABLES / "validation_anses.csv", index=False)
validation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, n in zip(axes, ["aet", "calcium"]):
    ax.scatter(comparaison[n + "_anses"], comparaison[n], s=6, alpha=0.4, color=C.PALETTE[0])
    lim = [0, comparaison[n].max()]
    ax.plot(lim, lim, "k--", lw=1)
    ax.set(title=f"{C.NUTRIMENTS[n][0]} ({C.NUTRIMENTS[n][1]}/j)", xlabel="Anses", ylabel="notre calcul")
plt.tight_layout()
plt.savefig(C.DOSSIER_FIGURES / "03_validation_anses.png", bbox_inches="tight")
plt.show()

**Conclusion** : nos apports sont **identiques** à ceux de l'Anses pour les 9 nutriments et les 2 121 adultes. Le pipeline (lecture, sélection, formule, prise en compte de `R24_pond`) est donc validé.

À noter : si on utilise `qte_conso` au lieu de `qte_conso_pond`, les écarts atteignent 1 à 7 % selon les nutriments. La pondération semaine / week-end compte vraiment.

➡️ Suite : `04_analyses.ipynb`